# Relay — GPUでLLM/VLMを起動する

Linux + CUDA GPUのJupyter/Colab用。Genesisと操作画面は手元で動かし、このノートブックはOpenAI互換の生成APIを提供します。CaP-X/GenesisをGPU側にインストールする必要はありません。

例はQwen2.5-VL-7B-Instruct。Relay本体はモデル非依存で、画像対応のChat Completions APIなら差し替え可能です。APIキーはノートブックの出力やGitに保存しません。

このvLLM起動レシピは接続用の例です。実モデルの検証結果と、検証したランタイムは `docs/verification/genesis-llm-20261102.md` を参照してください。


In [ ]:
import sys, platform, subprocess
assert platform.system() == "Linux", "Linux + CUDA GPUランタイムを選択してください"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print(sys.version)


## 1. 推論専用環境

既存のJupyter kernelやGenesisの依存を書き換えず、専用venvにvLLMを入れます。初回はモデルのダウンロードも必要です。VRAMに余裕のあるGPU（例：L4 24GB）を想定しています。


In [ ]:
from pathlib import Path
import urllib.request
ENV = Path.cwd() / ".relay-llm-venv"
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(ENV)], check=True)
    bootstrap = ENV / "get-pip.py"
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", bootstrap)
    subprocess.run([str(PYTHON), str(bootstrap)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "vllm==0.11.0"], check=True)


## 2. モデルと認証

手元の `CAPX_LLM_API_KEY` と同じ値を入力してください。入力内容は表示しません。キーを変更したときはAPIサーバーも再起動します。


In [ ]:
import getpass, os
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
API_KEY = getpass.getpass("Relay用のAPIキー: ")
assert len(API_KEY) >= 24, "24文字以上のキーを使ってください"


In [ ]:
if "llm_process" in globals() and llm_process.poll() is None:
    raise RuntimeError("既存のLLMプロセスを最後のセルで停止してから起動してください")
server_env = os.environ.copy()
server_env["VLLM_API_KEY"] = API_KEY
log = open("relay-llm.log", "w")
llm_process = subprocess.Popen([
    str(ENV / "bin/vllm"), "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--max-model-len", "8192", "--gpu-memory-utilization", "0.85",
    "--limit-mm-per-prompt", '{"image":2}',
], env=server_env, stdout=log, stderr=subprocess.STDOUT)
print("起動中。次のセルで準備完了を確認します。")


In [ ]:
import time, json
for _ in range(180):
    if llm_process.poll() is not None:
        raise RuntimeError("起動失敗。relay-llm.logを確認してください（共有前に個人情報を確認）。")
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models", headers={"Authorization": "Bearer " + API_KEY})
        with urllib.request.urlopen(req, timeout=3) as response:
            print("API準備完了:", [m["id"] for m in json.load(response)["data"]])
        break
    except (OSError, ValueError):
        time.sleep(5)
else:
    raise TimeoutError("準備が完了していません。relay-llm.logを確認してください")


## 3. 手元への接続

**GPUサーバーのJupyter**：手元のターミナルで `ssh -N -L 8000:127.0.0.1:8000 <自分のGPUサーバー>` を実行します。RelayのURLは `http://127.0.0.1:8000/v1/chat/completions`。

**Colab**：SSHがない場合だけ、以下の任意セルでCloudflare Quick Tunnelを作ります。画像・指示・生成コードがトンネルを通ります。APIキー認証は有効なままです。URLは一時的で、プロセス終了時に無効になります。商用APIを使う場合はこのノートブックの起動は不要です。


In [ ]:
# Colabで外部接続が必要な場合のみ True にする。
ENABLE_COLAB_TUNNEL = False
if ENABLE_COLAB_TUNNEL:
    import re
    if "tunnel_process" in globals() and tunnel_process.poll() is None:
        raise RuntimeError("既存のトンネルを最後のセルで停止してください")
    binary = Path.cwd() / "cloudflared"
    if not binary.exists():
        arch = "arm64" if platform.machine() == "aarch64" else "amd64"
        urllib.request.urlretrieve(f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}", binary)
        binary.chmod(0o700)
    tunnel_log = open("relay-tunnel.log", "w")
    tunnel_process = subprocess.Popen([str(binary), "tunnel", "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        matches = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", Path("relay-tunnel.log").read_text())
        if matches:
            print("Relayの --llm-url:", matches[-1] + "/v1/chat/completions")
            break
        time.sleep(1)
    else:
        raise TimeoutError("トンネルのURLを取得できません。relay-tunnel.logを確認してください")


## 4. 手元でGenesisを起動

リポジトリのターミナルで実行します。APIキーは環境変数から読み込みます。`--llm-url` は上で接続したURLに変更してください。

```bash
read -s CAPX_LLM_API_KEY; export CAPX_LLM_API_KEY
.local/genesis-sim/bin/python -m capx.extensions.hybrid.server \
  --backend genesis --backend-config configs/relay/genesis-panda.json \
  --no-vla --llm-url http://127.0.0.1:8000/v1/chat/completions \
  --llm-model Qwen/Qwen2.5-VL-7B-Instruct --llm-images \
  --port 8877 --output output/genesis-llm-01
```

操作画面の「LLMへ指示」に「現在位置から手先を上に2cm動かして、グリッパーを閉じてください」などを入力し「指示を実行」を押します。生成コードと結果を画面で確認できます。Escで停止、Hで手動へ。巻き戻し後も新しい指示で再開できます。

表示の「API設定済み」は接続設定の存在を示します。実際のリクエストが成功したことは生成コードと実行結果で確認してください。Genesisのsandboxにはタスク成功判定がありません。


## 5. 終了

手元のRelayをEscで停止して記録を保存してから、このセルで推論とトンネルを停止します。Colab/GPUサーバーのランタイム停止も忘れずに。


In [ ]:
for name in ("tunnel_process", "llm_process"):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
API_KEY = None
print("このノートブックで起動したプロセスを停止しました")


公式資料：[vLLM OpenAI互換API](https://docs.vllm.ai/en/v0.11.0/serving/openai_compatible_server.html)、[Qwen2.5-VLモデル](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct)、[Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)。
